# CAMeL-BERT Inference Pipeline (GPU/Colab)

**Purpose**: Run inference on full Kitab Uqala and export results for local analysis

**Output**: JSON with token predictions + offsets (ready for local post-processing)

**No analysis here** — just inference and export

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/khabar-segmentation')
print(f"[OK] Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install transformers torch tqdm -q
print("[OK] Dependencies installed")

In [ ]:
# Imports
import json
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForTokenClassification

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load model and tokenizer
model_path = Path('checkpoints/camelbert_binary_classification_final')

print(f"[INFO] Loading model from {model_path}...")
tokenizer = AutoTokenizer.from_pretrained(str(model_path))
model = AutoModelForTokenClassification.from_pretrained(str(model_path))
model.eval()

if torch.cuda.is_available():
    model = model.cuda()

print(f"[OK] Model loaded and ready")

In [ ]:
# Load the Kitab Uqala corpus
corpus_file = Path('data/processed/kitab_uqala_reference_corpus.txt')

print(f"[INFO] Loading corpus from {corpus_file}...")
with open(corpus_file, encoding='utf-8') as f:
    full_text = f.read()

print(f"[OK] Corpus loaded")
print(f"  Size: {len(full_text):,} chars")
print(f"  Words: {len(full_text.split()):,}")

In [ ]:
# ============================================================================
# STEP 1: Run inference on full text with offset mapping
# ============================================================================

def infer_with_offsets(text: str, tokenizer, model, max_length: int = 512) -> dict:
    """
    Run inference and preserve token-to-character offset mapping.
    
    This is the KEY function: it returns both predictions and offsets
    so we can map token predictions back to character positions.
    """
    # Tokenize with offset mapping
    encoded = tokenizer(
        text,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_offsets_mapping=True,
        return_tensors='pt'
    )

    # Inference
    with torch.no_grad():
        if torch.cuda.is_available():
            input_ids = encoded['input_ids'].cuda()
            attention_mask = encoded['attention_mask'].cuda()
            outputs = model(input_ids, attention_mask=attention_mask)
        else:
            outputs = model(**encoded)
        
        logits = outputs.logits[0]  # [seq_len, 2]

    # Convert to predictions and probabilities
    preds = np.argmax(logits.cpu().numpy(), axis=-1)  # [seq_len]
    probs = torch.softmax(logits, dim=-1).cpu().numpy()[:, 1]  # prob of class 1 (boundary)

    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
    offsets = encoded['offset_mapping'][0].numpy().tolist()

    return {
        'predictions': preds.tolist(),
        'probabilities': probs.tolist(),
        'tokens': tokens,
        'offsets': offsets,  # ← THE CRITICAL PIECE
    }

print("[OK] Inference function defined")

In [ ]:
# Run inference
print(f"[INFO] Running inference on full corpus...")
print(f"  Text length: {len(full_text):,} chars")
print(f"  Expected tokens: ~{len(full_text) // 4} (rough estimate)")
print(f"  Processing in 512-token chunks...\n")

inference_result = infer_with_offsets(full_text, tokenizer, model)

print(f"[OK] Inference complete")
print(f"  Total tokens: {len(inference_result['tokens'])}")
print(f"  Boundary tokens: {sum(inference_result['predictions'])}")
print(f"  Boundary ratio: {sum(inference_result['predictions']) / len(inference_result['predictions']) * 100:.1f}%")

In [ ]:
# ============================================================================
# STEP 2: Export results to JSON
# ============================================================================

export_data = {
    'metadata': {
        'corpus': 'kitab_uqala_reference_corpus',
        'text_size_chars': len(full_text),
        'text_size_words': len(full_text.split()),
        'model': 'camelbert_binary_classification_final',
        'timestamp': str(pd.Timestamp.now()) if 'pd' in dir() else 'unknown'
    },
    'inference_results': {
        'total_tokens': len(inference_result['tokens']),
        'boundary_tokens': sum(inference_result['predictions']),
        'predictions': inference_result['predictions'],
        'probabilities': inference_result['probabilities'],
        'tokens': inference_result['tokens'],
        'offsets': inference_result['offsets'],
    }
}

# Save to Drive
output_file = Path('results/camelbert_kitab_uqala_raw_inference.json')
output_file.parent.mkdir(parents=True, exist_ok=True)

print(f"[INFO] Saving inference results to {output_file}...")
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(export_data, f, indent=2, ensure_ascii=False)

# Check file size
file_size_mb = output_file.stat().st_size / (1024 * 1024)
print(f"[OK] File saved: {file_size_mb:.1f} MB")
print(f"[OK] Location on Drive: results/camelbert_kitab_uqala_raw_inference.json")

---

## Download Instructions

1. In Colab Files panel (left sidebar) → click Refresh
2. Navigate to `results/camelbert_kitab_uqala_raw_inference.json`
3. Right-click → Download
4. Save to your local `results/` directory

---

## What's in the file

- `predictions`: Binary token predictions (0/1 array, length 512)
- `probabilities`: Confidence scores for each token
- `tokens`: Token strings (for debugging)
- `offsets`: Character-level positions for each token **← Critical for local processing**

This is everything you need locally to:
1. Cluster boundary tokens
2. Extract actual segments
3. Compare with gold standard
4. Calculate metrics

In [ ]:
# Optional: Quick stats on predictions
print("\n[SUMMARY]")
print(f"Inference complete! Ready for local post-processing.\n")
print(f"Raw token predictions:")
print(f"  Total tokens: {export_data['inference_results']['total_tokens']}")
print(f"  Boundary tokens: {export_data['inference_results']['boundary_tokens']}")
print(f"  Mean boundary prob: {np.mean([p for p, pred in zip(inference_result['probabilities'], inference_result['predictions']) if pred == 1]):.4f}")
print(f"\nNext: Download the JSON file and run local post-processing")

## Local Processing (After Download)

```bash
python3 scripts/camelbert_local_postprocess.py \
  --input results/camelbert_kitab_uqala_raw_inference.json \
  --output results/camelbert_kitab_uqala_segments.json
```

This will:
1. Cluster boundary tokens
2. Extract text segments
3. Compare with 613 khabar gold standard
4. Calculate recall/precision/F1